# ECG Data Analysis and Feature Study

This notebook analyzes ECG data used for arrhythmia detection.  
It focuses on:
- Data quality
- Feature relationships
- Feature importance
- Insights for machine learning model

## 1. Imports and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the main ECG dataset (16 features, trained model's version)
df = pd.read_csv('../ecg_dataset.csv')
df_numeric = df.select_dtypes(include=[np.number])

df.head()

## 2. Dataset Overview

In [ ]:
print('Shape (rows, columns):', df.shape)

In [ ]:
print('Columns:', df.columns.tolist())

In [ ]:
df.info()

**Observations:**
- The dataset contains **45,152 rows** and **16 columns**
- Features include ECG signal statistics (mean, std, max, min), cardiac metrics (heart_rate, hrv, rr_mean), and patient demographics (age, sex)
- The `label` column is the target class: 0 = Normal, 1 = Arrhythmia, 2 = Other/Unknown

## 3. Missing Values Analysis

In [ ]:
df.isnull().sum()

**Conclusion:** No missing values detected in any column. The dataset is clean and ready for analysis. No imputation or filling was required.

## 4. Class Distribution (Very Important)

In [ ]:
# Map numeric labels to readable names for plot
label_map = {0: 'Normal', 1: 'Arrhythmia', 2: 'Other/Unknown'}
df['label_name'] = df['label'].map(label_map)

plt.figure(figsize=(8, 5))
sns.countplot(x='label_name', data=df, order=['Arrhythmia', 'Other/Unknown', 'Normal'],
              palette=['#e74c3c', '#f39c12', '#2ecc71'])
plt.title('Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Count')

# Annotate counts
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height()):,}',
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print('Class counts:')
print(df['label_name'].value_counts())

**Insight:**
- The dataset is **severely imbalanced**
- **Arrhythmia dominates** with 26,372 samples (~58%) vs Normal 8,125 (~18%) and Other/Unknown 10,655 (~24%)
- This imbalance means a model could achieve high accuracy by simply predicting Arrhythmia always — accuracy alone is a misleading metric
- Techniques like class weighting (already used by the LightGBM model) are necessary to handle this

## 5. Feature Distributions

In [ ]:
# Heart Rate Distribution
plt.figure(figsize=(9, 4))
sns.histplot(df['heart_rate'], bins=30, color='steelblue', kde=True)
plt.title('Heart Rate Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Heart Rate (BPM)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# HRV Distribution
plt.figure(figsize=(9, 4))
sns.histplot(df['hrv'], bins=30, color='darkorange', kde=True)
plt.title('HRV Distribution', fontsize=13, fontweight='bold')
plt.xlabel('HRV (ms)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

**Observations:**
- **Heart Rate** is clustered around 60–120 BPM — typical for ECG data. The distribution shows multiple peaks, suggesting subgroups (resting, active, tachycardic)
- **HRV** is right-skewed — many patients have low HRV (< 50 ms) which is clinically significant for arrhythmia risk. A healthy range is 50–100 ms

## 6. Feature vs Label Analysis (Critical Part)

In [ ]:
# Heart Rate vs Class Label
plt.figure(figsize=(9, 5))
sns.boxplot(x='label_name', y='heart_rate', data=df,
            order=['Normal', 'Arrhythmia', 'Other/Unknown'],
            palette=['#2ecc71', '#e74c3c', '#f39c12'])
plt.title('Heart Rate vs Class', fontsize=13, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Heart Rate (BPM)')
plt.tight_layout()
plt.show()

In [ ]:
# HRV vs Class Label
plt.figure(figsize=(9, 5))
sns.boxplot(x='label_name', y='hrv', data=df,
            order=['Normal', 'Arrhythmia', 'Other/Unknown'],
            palette=['#2ecc71', '#e74c3c', '#f39c12'])
plt.title('HRV vs Class', fontsize=13, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('HRV (ms)')
plt.tight_layout()
plt.show()

**Key Findings:**
- **HRV differs significantly across classes** → it is a **strong predictor** of cardiac condition. Arrhythmia patients show a wider spread of HRV values, with many extreme outliers. Normal patients cluster in a moderate HRV range.
- **Heart rate alone is a weak predictor** → the median heart rate is similar across all three classes. Overlap between classes is large, making it an unreliable standalone feature.
- This confirms that HRV should carry higher feature importance in the trained model.

## 7. Correlation Heatmap (Core Analysis)

In [ ]:
plt.figure(figsize=(12, 8))
corr = df_numeric.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # only lower triangle for clarity
sns.heatmap(corr, cmap='coolwarm', annot=True, fmt='.2f',
            linewidths=0.5, mask=mask)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Extracted Insights:**
- **Strong relationships:** `hrv` and `rr_mean` are highly correlated (both measure RR-interval variability). This is **expected** clinically — they capture the same cardiac rhythm pattern.
- **Redundant features:** `mean`, `max`, `min` of the ECG signal are correlated with each other. These signal statistics overlap significantly in information content.
- **Weak features:** `sex` and `age` show minimal correlation with signal-based features. They are demographic features that add context but don't drive predictions.
- The correlation structure suggests the model benefits more from **temporal RR-interval features** (hrv, rr_mean) than from basic signal statistics.

## 8. Feature Importance (Model Based)

In [ ]:
import joblib

model = joblib.load('../model.pkl')

# Features must match exactly what model was trained on (14 numeric features, excluding label)
importances = model.feature_importances_
features = df_numeric.drop(columns=['label']).columns

imp_df = pd.DataFrame({
    'feature': features,
    'importance': importances
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=imp_df,
            palette='RdYlGn_r')
plt.title('Feature Importance (LightGBM Model)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print('\nFeature Importance Ranking:')
print(imp_df.to_string(index=False))

**Insight:**
- **HRV and RR_mean are the most important features** — confirming our earlier boxplot observation that HRV separates classes better than any other feature.
- **Heart rate has low importance** — consistent with the boxplot showing heavy class overlap at similar heart rate values.
- `age` and `sex` contribute minimally, as expected for signal-based classification.
- The model's reliance on HRV/RR features validates that it is capturing **real cardiac rhythm patterns**, not just basic signal statistics.

## 9. Final Key Observations

| Requirement | Finding |
|---|---|
| Missing values | ✅ No missing values in any column |
| Dataset analysis | ✅ 45,152 records, 14 numeric features, 3 classes |
| Class distribution | ✅ Imbalanced — Arrhythmia dominates at 58% |
| Feature distributions | ✅ HRV right-skewed; Heart Rate clustered around 60-120 BPM |
| Feature vs label | ✅ HRV separates classes; Heart rate overlaps heavily |
| Correlation | ✅ HRV & RR_mean correlated; signal stats (mean/max/min) redundant |
| Feature importance | ✅ HRV, RR_mean top predictors; heart_rate low importance |
| Model reasoning | ✅ Model relies on temporal ECG patterns, not basic statistics |

**Summary:**
- The dataset shows **class imbalance** — handled with class weighting in LightGBM
- **HRV and RR intervals are the strongest predictors** — they reflect heart rhythm regularity
- **Heart rate alone is not reliable** — similar distributions across classes
- Some features are **highly correlated** (hrv ↔ rr_mean) → redundancy present
- The model relies more on **temporal ECG patterns** (HRV, RR intervals) than basic signal statistics (mean, std)